# Block-importance format selector

This notebook chooses one 4-bit format for the weights `W`, using calibration inputs `X_cal`. The aim is to keep the multiplication result close to `W @ X_cal`.

Instead of testing every weight at first, the selector tests important blocks of weights. It compares that choice with full-matrix evaluations later. The following sections explain each step.


## 1. Settings and inputs

Set the weight and calibration-input paths. `THRESHOLD` sets how much total importance the selected blocks should cover. `TIE_MODE` controls what happens when leading formats have almost equal scores.

The next cell loads and checks the inputs and sets the format and search options.


In [1]:
import itertools
import math
import os
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

W_PATH = "c:/Users/Admin/OneDrive/Desktop/System/W_C.npy"
X_CAL_PATH = "c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy"

THRESHOLD = 0.5
TIE_MODE = "refine"
METHODS = ["wt_x"]
RUN_SCALE_SEARCH = True

REF_FORMATS = ["E2M1", "E1M2", "MXUINT4", "MXINT4", "BFP4"]
REF_WINDOW = 3
REF_WINDOW_MAX = 5
REF_MAX_ELEMS = 2_000_000
REF_RANDOM_STARTS = 4
REF_PAIR_MOVES = True
REF_USE_GRAM_FACTOR = True
REF_SEED = 0

ASSIGNED_ELEMENT_BITS = 4

MAX_BITS_PER_WEIGHT = None

E8M0_MIN_EXP = -127
E8M0_MAX_EXP = 127

BFP_EXP_POLICY = "autoflex_predict"
BFP_ALPHA = 1.0

BFP_SHARING = "block"
BFP_BLOCK_SIZE = 32

FORMAT_CONFIGS = {
    "E2M1": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|2|1]",
        "grid": np.array(
            [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0],
            dtype=np.float64
        ),
        "baseline_exp_shift": -2,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "classroom/OCP-MX convention used in this project",
    },

    "E1M2": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|1|2]",
        "grid": np.array(
            [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5],
            dtype=np.float64
        ),
        "baseline_exp_shift": -1,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "adopted classroom-consistent E1M2 convention",
    },

    "MXINT4": {
        "kind": "mxint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|I|F] = [1|1|2]",
        "qmin": -8,
        "qmax": 7,
        "element_scale": 0.25,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; element precision scale 2^-2",
        "exceptions": "signed integer codes -8..7; no NaN/Inf codes",
        "convention_status": "adopted from the classroom MXINT construction",
    },

    "BFP4": {
        "kind": "bfp",
        "block_size": BFP_BLOCK_SIZE,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "flex4+8: 4-bit two's-complement integer mantissa (codes -8..7) + shared exponent",
        "mantissa_bits": 4,
        "qmin": -8,
        "qmax": 7,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "kappa = 2^E, value = m * kappa; 8-bit shared exponent per group (paper: M-bit exponent shared across the tensor)",
        "exceptions": "finite two's-complement integers only; saturate to [-8,7]; no NaN/Inf codes",
        "convention_status": "Flexpoint (Koster et al., arXiv:1711.02213): N-bit two's-complement mantissa, shared exponent, Autoflex exponent rule; group size / exponent width / rounding are course-level choices (see BFP_* switches)",
    },

    "MXUINT4": {
        "kind": "mxuint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "Unsigned 4-bit code; fixed midpoint zero point q_u - 8",
        "shared_scale": "power-of-two 2^e; fixed zero point 8; no per-block offset",
        "exceptions": "finite values; unsigned codes 0..15 decode to signed integers -8..7",
        "convention_status": "custom midpoint-coded unsigned-storage convention; no FP32 offset overhead",
    },
}

FORMAT_ORDER = list(REF_FORMATS)

if not os.path.isfile(W_PATH):
    raise FileNotFoundError(f"Weight file not found: {W_PATH}")
if not os.path.isfile(X_CAL_PATH):
    raise FileNotFoundError(f"Calibration file not found: {X_CAL_PATH}")
W_input = np.load(W_PATH, allow_pickle=False)
X_cal_input = np.load(X_CAL_PATH, allow_pickle=False)

for _name, _a in (("W", W_input), ("X_cal", X_cal_input)):
    if _a.ndim != 2 or min(_a.shape) == 0:
        raise ValueError(f"{_name} must be a nonempty 2-D array, got {_a.shape}")
    if not np.issubdtype(_a.dtype, np.number) or np.iscomplexobj(_a) or not np.isfinite(_a).all():
        raise ValueError(f"{_name} must be real, finite and numeric.")
if W_input.shape[1] != X_cal_input.shape[0]:
    raise ValueError(f"Need W.shape[1] == X_cal.shape[0], got {W_input.shape} and {X_cal_input.shape}.")
if not 0 < THRESHOLD <= 1:
    raise ValueError("THRESHOLD must be in (0, 1].")
if TIE_MODE not in ("refine", "stable"):
    raise ValueError("TIE_MODE must be 'refine' or 'stable'.")
if METHODS != ["wt_x"]:
    raise ValueError("This notebook is configured to run only the wt_x selector.")

W = np.asarray(W_input, dtype=np.float64)
X_cal = np.asarray(X_cal_input, dtype=np.float64)
Y_ref = W @ X_cal


## 2. Prepare the number formats

The next cell defines how five 4-bit formats round weights: E2M1, E1M2, MXUINT4, MXINT4, and BFP4.

Weights are grouped into row-wise blocks of up to 32. Each block shares a scale, which sets the spacing between allowed values. The baseline exponent is the starting choice of scale. Values are rounded to an allowed level and clipped if they exceed the format's range.

MXUINT4 uses stored codes 0–15 with a fixed midpoint of 8: subtracting 8 when decoding allows negative values too. The code also calculates storage, including the shared scale information.


In [2]:
W64 = np.asarray(W, dtype=np.float64)
X64 = np.asarray(X_cal, dtype=np.float64)
Y64 = np.asarray(Y_ref, dtype=np.float64)
WEIGHT_ENERGY = float(np.sum(W64 ** 2))
OUTPUT_ENERGY = float(np.sum(Y64 ** 2))

def rowwise_ranges(cols, block_size):

    for start in range(0, cols, block_size):
        yield start, min(start + block_size, cols)

def count_rowwise_blocks(W, block_size):
    return W.shape[0] * math.ceil(W.shape[1] / block_size)

def storage_for_format(fmt):
    cfg = FORMAT_CONFIGS[fmt]
    if fmt == "BFP4" and BFP_SHARING == "tensor":
        n_blocks = 1
    else:
        n_blocks = count_rowwise_blocks(W64, cfg["block_size"])
    element_bits = cfg["element_bits"] * W64.size
    metadata_bits = cfg["metadata_bits"] * n_blocks
    total_bits = element_bits + metadata_bits
    return {
        "Blocks": n_blocks,
        "Element_Bits": element_bits,
        "Metadata_Bits": metadata_bits,
        "Total_Storage_Bits": total_bits,
        "Effective_Bits_per_Weight": total_bits / W64.size,
    }

def nearest_grid_rne(values, grid):
    values = np.asarray(values, dtype=np.float64)
    out = np.empty_like(values)

    for i, x in enumerate(values):
        d = np.abs(grid - x)
        dmin = np.min(d)
        candidates = np.where(np.isclose(d, dmin, rtol=0.0, atol=1e-15))[0]
        if len(candidates) == 1:
            idx = int(candidates[0])
        else:
            even = candidates[candidates % 2 == 0]
            idx = int(even[0] if len(even) else candidates[0])
        out[i] = grid[idx]
    return out

def _bfp_gamma(block, E):
    """Gamma of the paper: max |mantissa| produced at scale kappa=2^E,
    saturated at 2^(N-1)-1 (the paper's overflow test is Gamma >= 2^(N-1)-1)."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    m = np.rint(np.abs(np.asarray(block, dtype=np.float64)) / np.exp2(float(E)))
    return float(min(np.max(m), 2 ** (N - 1) - 1))

def bfp_exp_autoflex_predict(block):
    """Paper Algorithm 2 (line 9-10) for a static tensor: one observation,
    std(f)=0, no previous kappa, alpha=BFP_ALPHA.   E = ceil(log2 chi) - N + 1."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    chi = BFP_ALPHA * float(np.max(np.abs(block)))
    if chi == 0:
        return 0
    return int(np.ceil(np.log2(chi))) - N + 1

def bfp_exp_autoflex_init(block):
    """Paper Algorithm 1 (trial-and-error initialisation), literal.
    kappa = 2^E starts at 1 (E=0). Overflow: kappa *= 2^floor((N-1)/2).
    Underflow (Gamma < 2^(N-2)): kappa *= 2^(ceil(log2 max(Gamma,1)) - (N-2)),
    accepted only if Gamma > 2^(floor((N-1)/2)-2) ('enough bits for a reliable jump')."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    if float(np.max(np.abs(block))) == 0:
        return 0
    E = 0

    for _ in range(1024):
        G = _bfp_gamma(block, E)
        if G >= 2 ** (N - 1) - 1:
            E += (N - 1) // 2
        elif G < 2 ** (N - 2):
            E += int(np.ceil(np.log2(max(G, 1.0)))) - (N - 2)
            if G > 2.0 ** ((N - 1) // 2 - 2):
                break
        else:
            break
        E = int(np.clip(E, E8M0_MIN_EXP, E8M0_MAX_EXP))
    return E

def bfp_shared_exponent(block):
    if BFP_EXP_POLICY == "autoflex_init":
        return bfp_exp_autoflex_init(block)
    if BFP_EXP_POLICY == "autoflex_predict":
        return bfp_exp_autoflex_predict(block)
    raise ValueError(f"Unknown BFP_EXP_POLICY={BFP_EXP_POLICY}")

def baseline_exp(block, fmt):
    amax = float(np.max(np.abs(block)))
    if amax == 0:
        return 0

    raw = int(np.floor(np.log2(amax)))
    cfg = FORMAT_CONFIGS[fmt]

    if cfg["kind"] == "mxfp":
        exp = raw + cfg["baseline_exp_shift"]
    elif cfg["kind"] == "mxuint":

        neg_extent = max(0.0, -float(np.min(block)))
        pos_extent = max(0.0, float(np.max(block)))
        required_step = max(neg_extent / 8.0, pos_extent / 7.0)
        exp = int(np.ceil(np.log2(required_step))) if required_step > 0 else 0
    elif cfg["kind"] == "bfp":

        exp = bfp_shared_exponent(block)
    else:
        exp = raw

    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))
    return exp

def quantize_block_at_exp(block, fmt, exp, return_events=False):
    block = np.asarray(block, dtype=np.float64)
    cfg = FORMAT_CONFIGS[fmt]

    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))

    zeroed = np.zeros(block.shape, dtype=bool)
    saturated = np.zeros(block.shape, dtype=bool)

    if cfg["kind"] == "mxfp":
        scale = np.exp2(exp)
        local = np.abs(block) / scale
        qlocal = nearest_grid_rne(local, cfg["grid"])
        qblock = np.sign(block) * qlocal * scale
        zeroed = (block != 0) & (qblock == 0)

        grid = cfg["grid"]
        if len(grid) >= 2:
            final_step = grid[-1] - grid[-2]
            sat_threshold = grid[-1] + 0.5 * final_step
            saturated = local > sat_threshold

    elif cfg["kind"] == "mxint":
        step = np.exp2(exp) * cfg["element_scale"]
        q_raw = np.rint(block / step)
        q = np.clip(q_raw, cfg["qmin"], cfg["qmax"])
        qblock = q * step
        zeroed = (block != 0) & (q == 0)
        saturated = (q_raw < cfg["qmin"]) | (q_raw > cfg["qmax"])

    elif cfg["kind"] == "bfp":

        scale = np.exp2(exp)
        m_raw = np.rint(block / scale)
        m = np.clip(m_raw, cfg["qmin"], cfg["qmax"])
        qblock = m * scale
        zeroed = (block != 0) & (m == 0)
        saturated = (m_raw < cfg["qmin"]) | (m_raw > cfg["qmax"])

    elif cfg["kind"] == "mxuint":

        scale = np.exp2(exp)
        q_raw = np.rint(block / scale) + 8
        q = np.clip(q_raw, 0, 15)
        q_signed = q - 8
        qblock = q_signed * scale
        zeroed = (block != 0) & (q_signed == 0)
        saturated = (q_raw < 0) | (q_raw > 15)

    else:
        raise ValueError(f"Numerical quantizer not defined for {fmt}")

    if return_events:
        return qblock, zeroed, saturated
    return qblock


## 3. Prepare blocks and count work

The next cell records each block's position and original weights. Helper functions quantize a block or the complete matrix and measure full output error.

For the first comparison, only selected blocks are quantized. Full-matrix work is needed later to create the chosen result, and may also be used to resolve a tie.

Work counters estimate arithmetic such as additions and multiplications. They do not include every operation, such as sorting or rounding, and are not measurements of runtime.


In [3]:
COUNT_KEYS = ["add", "sub", "mul", "div", "sqrt", "quantized_values", "quantized_blocks"]
_FLOPS_PER_VALUE = {"mxfp": dict(div=1, sub=8, mul=2), "mxint": dict(div=1, mul=1),
                    "bfp": dict(div=1, mul=1), "mxuint": dict(div=1, add=1, sub=1, mul=1)}

def new_counts():
    return dict.fromkeys(COUNT_KEYS, 0)

def add_counts(counts, **operations):
    if counts is not None:

        for name, number in operations.items():
            counts[name] += int(number)

def combine_counts(*parts):
    total = new_counts()

    for part in parts:

        for name in COUNT_KEYS:
            total[name] += part[name]
    return total

def flops(counts):
    return counts["add"] + counts["sub"] + counts["mul"] + counts["div"]

SELECTION_BLOCK_SIZE = FORMAT_CONFIGS[FORMAT_ORDER[0]]["block_size"]
assert all(FORMAT_CONFIGS[f]["block_size"] == SELECTION_BLOCK_SIZE for f in FORMAT_ORDER), \
    "All candidate formats must share one block size for block selection."
TENSOR_BFP_EXP = baseline_exp(W64.ravel(), "BFP4") if BFP_SHARING == "tensor" else None

def eligible_formats():
    """All five formats are defined for signed weights (MXUINT4 is midpoint-coded). Optional storage cap."""
    rows, eligible = [], []

    for fmt in FORMAT_ORDER:
        st = storage_for_format(fmt)
        status, reason = "ELIGIBLE", "Encoding defined in FORMAT_CONFIGS."
        if MAX_BITS_PER_WEIGHT is not None and st["Effective_Bits_per_Weight"] > MAX_BITS_PER_WEIGHT + 1e-12:
            status, reason = "OVER_STORAGE_LIMIT", "Effective storage exceeds MAX_BITS_PER_WEIGHT."
        if status == "ELIGIBLE":
            eligible.append(fmt)
        rows.append(dict(Format=fmt, Status=status, Reason=reason, **st))
    return eligible, pd.DataFrame(rows)

def prepare_blocks(W):
    """Row-wise blocks; a shorter final block per row is kept at its real length."""
    blocks = []

    for r in range(W.shape[0]):

        for start, end in rowwise_ranges(W.shape[1], SELECTION_BLOCK_SIZE):
            values = W[r, start:end]
            blocks.append(dict(index=len(blocks), row=r, start=start, end=end, values=values,
                               amax=float(np.max(np.abs(values)))))
    return blocks

def quantize_prepared(block, fmt, counts=None):
    """Quantize one complete block at its own baseline shared exponent."""
    n = len(block["values"])
    add_counts(counts, quantized_values=n, quantized_blocks=1)
    if block["amax"] == 0:
        return np.zeros(n)
    exponent = TENSOR_BFP_EXP if (fmt == "BFP4" and BFP_SHARING == "tensor") else baseline_exp(block["values"], fmt)
    per_value = _FLOPS_PER_VALUE[FORMAT_CONFIGS[fmt]["kind"]]
    add_counts(counts, **{k: v * n for k, v in per_value.items()})
    return quantize_block_at_exp(block["values"], fmt, exponent)

def quantize_matrix(W, fmt, blocks, counts=None, cache=None):
    """Quantize a whole matrix, re-using any block already quantized (cache)."""
    cache = {} if cache is None else cache
    out = np.zeros_like(W)

    for block in blocks:
        if block["index"] not in cache:
            cache[block["index"]] = quantize_prepared(block, fmt, counts)
        out[block["row"], block["start"]:block["end"]] = cache[block["index"]]
    return out

def output_rmse(W, Wq, X, counts=None):
    """Full calibration-output RMSE of (Wq - W) @ X."""
    error = (Wq - W) @ X
    m, n = W.shape
    p = X.shape[1]
    add_counts(counts, sub=m * n, mul=m * n * p + m * p, add=m * (n - 1) * p + m * p - 1, div=1, sqrt=1)
    return float(np.sqrt(np.mean(error * error)))


## 4. Select important blocks and choose a format

This is the main algorithm. It uses both weight size and input activity to decide which blocks deserve attention.

**Step 1 — Measure input activity.** Square the values in each input row and add them. A larger total means that feature has more energy in the calibration inputs.

**Step 2 — Give each weight an importance score.** Square the weight and multiply by the energy of its matching input feature. For example, a weight of 2 with input energy 3 has importance `2 × 2 × 3 = 12`. Large weights connected to active inputs get larger scores.

**Step 3 — Select whole blocks.** Rank blocks by the largest individual weight-importance score inside each block. Then take blocks in that order until the sum of their importance scores reaches `THRESHOLD` of the total. The ranking uses each block's maximum, while coverage uses its sum—these are different quantities.

**Example:** block A has scores `[8, 1]` and block B has `[6, 6]`. A comes first because its maximum, 8, exceeds B's maximum, 6. Their total importance is 21. At a threshold of 40%, A alone covers enough: `9 / 21 ≈ 43%`. At 50%, both blocks are needed. A 50% threshold does not mean selecting exactly half the weights.

**Step 4 — Score each format.** Quantize the selected blocks using scales from their complete original weights. Put their weight errors into a matrix, with zeros elsewhere, and multiply by `X_cal`. Square the resulting output errors, average them, and take the square root. This is RMSE: lower is better. Signed contributions are added before squaring, so selected-block errors can cancel or reinforce each other.

**Step 5 — Handle ties and build the result.** Scores within a small tolerance count as tied. If `TIE_MODE = "refine"`, compare the tied leading formats using full-output RMSE. Otherwise, use the configured format order. Finally, quantize the whole matrix with the chosen format, reusing already calculated blocks where possible.

The initial score includes only selected-block errors. Omitted blocks may change the best format when the complete output is checked. This cell also defines the full baseline, which tests every format on every block at its baseline scale.


In [4]:
TIE_RTOL = 1e-6
TIE_ATOL = 1e-15

def tied(a, b, counts=None):
    add_counts(counts, sub=1, mul=1, add=1)
    return abs(a - b) <= TIE_ATOL + TIE_RTOL * max(abs(a), abs(b))

def ranking_groups(scores, counts=None):
    """Equal scores share a rank; each group is compared with its first score."""
    ordered = sorted(scores, key=lambda fmt: scores[fmt])
    groups = []

    for fmt in ordered:
        if not groups or not tied(scores[fmt], scores[groups[-1][0]], counts):
            groups.append([fmt])
        else:
            groups[-1].append(fmt)

    for group in groups:
        group.sort(key=FORMAT_ORDER.index)
    return groups

def ranking_text(groups):
    return " < ".join(" = ".join(group) for group in groups)

# Rank by block maximum; stop using cumulative block sums.
def select_blocks(W, X, blocks, threshold, counts):
    """Whole blocks, highest priority first, until `threshold` of the total importance is reached."""
    activation_energy = np.sum(X * X, axis=1)
    importance = (W * W) * activation_energy[None, :]
    n, p = X.shape
    add_counts(counts, mul=n * p + 2 * W.size, add=n * (p - 1))
    block_importance, priority = [], []

    for block in blocks:
        importance_in_block = importance[block["row"], block["start"]:block["end"]]
        total = float(np.sum(importance_in_block))
        block_importance.append(total)
        add_counts(counts, add=len(importance_in_block) - 1)
        priority.append(float(np.max(importance_in_block)))
    total_energy = float(np.sum(block_importance))
    add_counts(counts, add=len(blocks) - 1)
    if total_energy == 0:
        return [], 0.0
    order = sorted(range(len(blocks)), key=lambda index: priority[index], reverse=True)
    add_counts(counts, mul=1)
    target = threshold * total_energy
    accumulated, selected = 0.0, []

    for index in order:
        selected.append(index)
        accumulated += block_importance[index]
        add_counts(counts, add=1)
        if accumulated >= target:
            break
    selected.sort()
    add_counts(counts, div=1)
    return selected, accumulated / total_energy

# Omitted blocks contribute zero error to this partial score.
def subset_scores(W, X, blocks, selected, formats, counts):
    """Quantize selected blocks and score their contribution to output error."""
    caches, scores = {}, {}
    m = W.shape[0]
    p = X.shape[1]
    weight_count = sum(blocks[i]["end"] - blocks[i]["start"] for i in selected)

    for fmt in formats:
        caches[fmt] = {}
        delta_selected = np.zeros_like(W)

        for index in selected:
            block = blocks[index]
            quantized = quantize_prepared(block, fmt, counts)
            caches[fmt][index] = quantized
            delta = quantized - block["values"]
            size = len(delta)
            add_counts(counts, sub=size)
            delta_selected[block["row"], block["start"]:block["end"]] = delta
            add_counts(counts, mul=size * p, add=size * p)
        if weight_count == 0:
            scores[fmt] = 0.0
        else:
            error = delta_selected @ X
            scores[fmt] = float(np.sqrt(np.mean(error * error)))
            add_counts(counts, mul=m * p, add=m * p - 1, div=1, sqrt=1)
    return scores, caches

# Resolve leading ties if requested, then build the full chosen matrix.
def run_selector(W, X, threshold=THRESHOLD, tie_mode=TIE_MODE):
    preparation, scoring, refinement, materialization = new_counts(), new_counts(), new_counts(), new_counts()
    blocks = prepare_blocks(W)
    formats, _ = eligible_formats()
    if not formats:
        raise ValueError("No eligible format satisfies the configured storage constraint.")
    selected, energy_fraction = select_blocks(W, X, blocks, threshold, preparation)
    scores, caches = subset_scores(W, X, blocks, selected, formats, scoring)
    groups = ranking_groups(scores, scoring)
    finalists = groups[0]
    raw_choice = chosen = finalists[0]
    final_ties = finalists.copy()
    full_matrices, refinement_scores = {}, {}
    reason = "Unique initial winner."
    if len(finalists) > 1:
        reason = "Stable format order used within a reported tie."
        if not selected:
            reason = "Zero importance: every candidate has zero calibration output error."
        elif tie_mode == "refine":

            for fmt in finalists:
                q = quantize_matrix(W, fmt, blocks, refinement, caches[fmt])
                full_matrices[fmt] = q
                refinement_scores[fmt] = output_rmse(W, q, X, refinement)
            final_ties = ranking_groups(refinement_scores, refinement)[0]
            chosen = final_ties[0]
            reason = "Tied finalists checked with full calibration output RMSE; cost included."
    Wq = full_matrices[chosen] if chosen in full_matrices else quantize_matrix(W, chosen, blocks, materialization, caches[chosen])
    selection_counts = combine_counts(preparation, scoring, refinement)
    return dict(method="wt_x", chosen=chosen, raw_choice=raw_choice, scores=scores, ranking=groups,
                top_ties=finalists, final_ties=final_ties, tie_reason=reason, refinement_scores=refinement_scores,
                selected_blocks=selected, n_blocks=len(blocks),
                selected_weights=sum(len(blocks[i]["values"]) for i in selected),
                energy_fraction=energy_fraction, Wq=Wq,
                raw_counts=combine_counts(preparation, scoring), refinement_counts=refinement,
                materialization_counts=materialization, selection_counts=selection_counts,
                production_counts=combine_counts(selection_counts, materialization))

def full_baseline(W, X):
    """Project brute force: quantize EVERY block of every format at the baseline exponent, score the full output."""
    counts = new_counts()
    blocks = prepare_blocks(W)
    formats, _ = eligible_formats()
    scores, matrices = {}, {}

    for fmt in formats:
        matrices[fmt] = quantize_matrix(W, fmt, blocks, counts)
        scores[fmt] = output_rmse(W, matrices[fmt], X, counts)
    return dict(scores=scores, ranking=ranking_groups(scores, counts), matrices=matrices, counts=counts)


## 5. Search for better block scales

This optional reference asks whether changing block scales improves the complete multiplication result.

For each format and row, the search builds nearby exponent choices for each block, tries combinations of these choices, and measures full output error. For example, two blocks with three choices each give `3 × 3 = 9` combinations. Checking blocks together matters because their output errors can cancel or add together.

The search is limited to the configured range around baseline exponents. Exact enumeration checks every combination in that range when it fits the search limits; helper routines also support fallback search when exact enumeration is too large. The reported search diagnostics distinguish these cases. This is not a search over every possible exponent.

These reference results evaluate the selector. They are not used by the block-importance selector to choose its format.


In [5]:
_reference_qcache = {}

def _gram_factor(X):
    """Return F such that F F^T = X X^T on the non-zero eigenspace."""
    G = X @ X.T
    lam, V = np.linalg.eigh(G)

    if lam.size == 0 or np.max(lam) <= 0:
        return np.zeros((X.shape[0], 0), dtype=np.float64)

    keep = lam > np.max(lam) * 1e-14
    return V[:, keep] * np.sqrt(lam[keep])

def _reference_candidates(fmt, r, b, s, e, window):
    cfg = FORMAT_CONFIGS[fmt]
    block = W64[r, s:e]
    base = baseline_exp(block, fmt)

    exps = []

    for off in sorted(
        range(-window, window + 1),
        key=lambda o: (abs(o), o)
    ):
        exp = base + off

        if "exp_min" in cfg:
            exp = int(
                np.clip(
                    exp,
                    cfg["exp_min"],
                    cfg["exp_max"]
                )
            )

        if exp not in exps:
            exps.append(exp)

    Q = []

    for exp in exps:
        key = (fmt, r, b, exp)

        if key not in _reference_qcache:
            _reference_qcache[key] = quantize_block_at_exp(
                block,
                fmt,
                exp
            )

        Q.append(
            _reference_qcache[key]
        )

    return (
        base,
        np.asarray(exps, dtype=int),
        np.asarray(Q, dtype=np.float64),
    )

def _enumerate_exact(D):
    S = D[0]

    for d in D[1:]:
        S = (
            S[:, None, :]
            + d[None, :, :]
        ).reshape(-1, S.shape[-1])

    sse = np.einsum(
        "ij,ij->i",
        S,
        S
    )

    k = int(
        np.argmin(sse)
    )

    shape = [
        len(d)
        for d in D
    ]

    idx = [
        int(i)
        for i in np.unravel_index(
            k,
            shape
        )
    ]

    return idx, float(sse[k])

def _reference_heuristic(
    D,
    rng,
    n_random,
    pair_moves
):
    """
    Multi-start coordinate descent, optionally followed by pair moves.
    Returns a strong feasible solution; it is not a proof of global optimality.
    """

    nb = len(D)
    K = [
        len(d)
        for d in D
    ]

    sq = [
        np.einsum(
            "ij,ij->i",
            d,
            d
        )
        for d in D
    ]

    starts = [
        [0] * nb,
        [
            int(np.argmin(s))
            for s in sq
        ]
    ]

    starts += [
        [
            int(rng.integers(k))
            for k in K
        ]
        for _ in range(
            n_random
        )
    ]

    best_idx = None
    best = np.inf

    for idx in starts:

        idx = list(idx)

        e = sum(
            D[b][idx[b]]
            for b in range(nb)
        )

        cur = float(
            e @ e
        )

        improved = True

        while improved:
            improved = False

            for b in range(nb):

                base = (
                    e
                    - D[b][idx[b]]
                )

                t = (
                    base[None, :]
                    + D[b]
                )

                s = np.einsum(
                    "ij,ij->i",
                    t,
                    t
                )

                k = int(
                    np.argmin(s)
                )

                if s[k] < cur * (1 - 1e-12):

                    idx[b] = k
                    e = t[k].copy()
                    cur = float(s[k])
                    improved = True

            if (
                not improved
                and pair_moves
            ):

                for b in range(nb):

                    for c in range(
                        b + 1,
                        nb
                    ):

                        base = (
                            e
                            - D[b][idx[b]]
                            - D[c][idx[c]]
                        )

                        t = (
                            base[None, None, :]
                            + D[b][:, None, :]
                            + D[c][None, :, :]
                        )

                        s = np.einsum(
                            "ijk,ijk->ij",
                            t,
                            t
                        )

                        i, j = np.unravel_index(
                            np.argmin(s),
                            s.shape
                        )

                        if s[i, j] < cur * (1 - 1e-12):

                            idx[b] = int(i)
                            idx[c] = int(j)
                            e = t[i, j].copy()
                            cur = float(s[i, j])
                            improved = True

        if cur < best:
            best = cur
            best_idx = list(idx)

    return best_idx, best

def _bfp_tensor_search(X):
    """Exhaustive search of the single tensor-wide Flexpoint exponent
    (window +/-REF_WINDOW around the Autoflex exponent, widened to REF_WINDOW_MAX
    if the optimum sits on the window edge). Objective: actual ||(W - Wq) X||_F^2."""
    t0 = time.perf_counter()
    cfg = FORMAT_CONFIGS["BFP4"]
    base = baseline_exp(W64.ravel(), "BFP4")
    window = REF_WINDOW

    while True:
        exps = sorted(
            {int(np.clip(base + o, cfg["exp_min"], cfg["exp_max"])) for o in range(-window, window + 1)},
            key=lambda e: (abs(e - base), e - base),
        )
        best = None

        for E in exps:
            Wq = quantize_block_at_exp(W64, "BFP4", E)
            R = (W64 - Wq) @ X
            s = float(np.sum(R * R))
            if best is None or s < best[0]:
                best = (s, E, Wq)
        edge = abs(best[1] - base) == window
        if edge and window < REF_WINDOW_MAX:
            window += 1
            continue
        break
    n = W64.shape[0]
    return best[2], {
        "exact_rows": n, "heuristic_rows": 0,
        "widened_rows": n if window > REF_WINDOW else 0,
        "edge_rows": n if edge else 0,
        "offsets": [int(best[1] - base)],
        "seconds": time.perf_counter() - t0,
        "method": "EXACT",
    }

def strong_reference_search(
    fmt,
    X,
    mode=None,
    n_random=None,
    pair_moves=None
):
    """
    mode=None:
        exact when feasible, stronger heuristic otherwise.

    mode='exact':
        force exact enumeration; raises if the memory/search cap is exceeded.

    mode='heuristic':
        force the heuristic search.

    The returned objective is an actual calibration-output SSE objective,
    evaluated efficiently through an exact Gram factor when enabled.

    This is exhaustive over the explicitly bounded exponent candidate window
    (initially +/-3, widened as needed up to +/-5 from the baseline exponent),
    not over every possible E8M0 exponent. The assignment-sized dataset is
    required to use exact enumeration for every row; a heuristic fallback
    raises an error rather than silently passing as brute force.
    """

    n_random = (
        REF_RANDOM_STARTS
        if n_random is None
        else n_random
    )

    pair_moves = (
        REF_PAIR_MOVES
        if pair_moves is None
        else pair_moves
    )

    if fmt == "BFP4" and BFP_SHARING == "tensor":
        return _bfp_tensor_search(X)

    F = (
        _gram_factor(X)
        if REF_USE_GRAM_FACTOR
        else X
    )

    bs = FORMAT_CONFIGS[
        fmt
    ]["block_size"]

    ranges = list(
        rowwise_ranges(
            W64.shape[1],
            bs
        )
    )

    nb = len(ranges)

    feature_dim = max(
        1,
        F.shape[1]
    )

    limit = max(
        1,
        REF_MAX_ELEMS
        // feature_dim
    )

    Wq = np.zeros_like(
        W64
    )

    info = {
        "exact_rows": 0,
        "heuristic_rows": 0,
        "widened_rows": 0,
        "edge_rows": 0,
        "offsets": [],
    }

    t0 = time.perf_counter()

    for r in range(
        W64.shape[0]
    ):

        T = [
            W64[r, s:e]
            @ F[s:e]
            for s, e in ranges
        ]

        rng = np.random.default_rng(
            REF_SEED + r
        )

        window = REF_WINDOW

        while True:

            C = [
                _reference_candidates(
                    fmt,
                    r,
                    b,
                    s,
                    e,
                    window
                )
                for b, (s, e)
                in enumerate(ranges)
            ]

            D = [
                T[b]
                - C[b][2] @ F[s:e]
                for b, (s, e)
                in enumerate(ranges)
            ]

            combinations = math.prod(
                len(d)
                for d in D
            )

            exact = (
                mode == "exact"
                or (
                    mode is None
                    and combinations <= limit
                )
            )

            if (
                exact
                and combinations > limit
            ):
                raise RuntimeError(
                    f"{fmt} row {r}: "
                    f"{combinations:,} combinations "
                    f"exceed exact-search cap {limit:,}"
                )

            if exact:
                idx, _ = _enumerate_exact(
                    D
                )
            else:
                idx, _ = _reference_heuristic(
                    D,
                    rng,
                    n_random,
                    pair_moves
                )

            offsets = [
                int(
                    C[b][1][idx[b]]
                    - C[b][0]
                )
                for b in range(nb)
            ]

            edge = any(
                abs(o) == window
                for o in offsets
            )

            can_widen_exact = (
                (
                    2 * (window + 1)
                    + 1
                ) ** nb
                <= limit
            )

            if (
                edge
                and window < REF_WINDOW_MAX
                and (
                    not exact
                    or can_widen_exact
                )
            ):
                window += 1
                continue

            break

        if exact:
            info[
                "exact_rows"
            ] += 1
        else:
            info[
                "heuristic_rows"
            ] += 1

        info[
            "widened_rows"
        ] += int(
            window > REF_WINDOW
        )

        info[
            "edge_rows"
        ] += int(edge)

        info[
            "offsets"
        ] += offsets

        for b, (s, e) in enumerate(
            ranges
        ):
            Wq[
                r,
                s:e
            ] = C[b][2][
                idx[b]
            ]

    info[
        "seconds"
    ] = (
        time.perf_counter()
        - t0
    )

    if info[
        "heuristic_rows"
    ] == 0:
        info[
            "method"
        ] = "EXACT"
    elif info[
        "exact_rows"
    ] == 0:
        info[
            "method"
        ] = "HEURISTIC (strong feasible upper bound)"
    else:
        info[
            "method"
        ] = "MIXED exact/heuristic"

    return Wq, info

def _actual_error_pct(
    Wq,
    Xs
):
    denominator = np.linalg.norm(
        W64 @ Xs
    )

    if denominator == 0:
        return 0.0

    return (
        100.0
        * np.linalg.norm(
            (W64 - Wq)
            @ Xs
        )
        / denominator
    )


## 6. Run the methods

Run the selector once. When enabled, also run the full baseline and scale-search reference for comparison.


In [6]:
t0 = time.perf_counter()
ELIGIBLE, ELIGIBILITY = eligible_formats()
RESULTS = {"wt_x": run_selector(W64, X64)}
BASE = full_baseline(W64, X64)

OPT = None
if RUN_SCALE_SEARCH:
    opt_scores, opt_matrices, opt_info = {}, {}, {}

    for fmt in ELIGIBLE:
        opt_matrices[fmt], opt_info[fmt] = strong_reference_search(fmt, X64, mode="exact")
        opt_scores[fmt] = output_rmse(W64, opt_matrices[fmt], X64)
    OPT = dict(scores=opt_scores, ranking=ranking_groups(opt_scores), info=opt_info)

REF_RMS = float(np.sqrt(np.mean(Y64 ** 2)))
print(f"Done in {time.perf_counter() - t0:.2f} s  |  W {W64.shape}  X_cal {X64.shape}  |  "
      f"{len(prepare_blocks(W64))} blocks of <= {SELECTION_BLOCK_SIZE} weights")


Done in 12.33 s  |  W (100, 100)  X_cal (100, 256)  |  400 blocks of <= 32 weights


## 7. Read the results

The following cells build tables comparing the selector with the reference methods.

- **Selector score:** output RMSE caused by selected blocks only.
- **Actual output error (%):** full output error relative to the size of the original `W @ X_cal` output.
- **Rank:** 1 is best; formats treated as tied share a rank.

Lower error is better, but selected-block RMSE and full-output error percentage are different measurements. Use the rankings and chosen formats to compare decisions.


In [7]:
_GOOD = "background-color:#d4edda;color:#155724;font-weight:bold"
_BAD = "background-color:#f8d7da;color:#721c24;font-weight:bold"

def _best_in_row(tol=1e-9):

    def f(row):
        vals = pd.to_numeric(row, errors="coerce")
        m = vals.min()
        return [_GOOD if (pd.notna(v) and abs(v - m) <= tol * max(1.0, abs(m))) else "" for v in vals]
    return f

def _flag(v):
    if isinstance(v, str) and v.startswith("✅"):
        return _GOOD
    if isinstance(v, str) and v.startswith("❌"):
        return _BAD
    return ""

def show_matrix(df, title, fmt=None, best=None, flag_cols=None):
    """Styled matrix. best='min' highlights the row-wise best (smallest) cell(s) in green;
    flag_cols colour the check-mark columns. Falls back to a plain table if styling is unavailable."""

    try:
        sty = df.style.set_caption(title)
        if fmt is not None:
            sty = sty.format(fmt, na_rep="-")
        sty = sty.set_table_styles([
            {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"),
                                              ("text-align", "left"), ("padding", "6px 0")]},
            {"selector": "th", "props": [("text-align", "center"), ("background-color", "#eef1f6"),
                                         ("color", "#111")]},
            {"selector": "td", "props": [("text-align", "center")]},
        ])
        if best == "min":
            sty = sty.apply(_best_in_row(), axis=1)
        if flag_cols:
            mapper = getattr(sty, "map", None) or sty.applymap
            sty = mapper(_flag, subset=flag_cols)
        display(sty)
    except Exception:
        print(title)
        display(df)


In [8]:
from itertools import combinations

idx_fmt = pd.Index(ELIGIBLE, name="Format")

def group_ranks(groups):
    """Competition ranking (1,1,3,...) from tie groups."""
    ranks, position = {}, 1

    for group in groups:

        for fmt in group:
            ranks[fmt] = position
        position += len(group)
    return ranks

def to_pct(rmse):
    return 100.0 * rmse / REF_RMS if REF_RMS > 0 else 0.0

base_rank = group_ranks(BASE["ranking"])
opt_rank = group_ranks(OPT["ranking"]) if OPT else None
base_err = {f: to_pct(BASE["scores"][f]) for f in ELIGIBLE}
opt_err = {f: to_pct(OPT["scores"][f]) for f in ELIGIBLE} if OPT else None
method_rank = {m: group_ranks(RESULTS[m]["ranking"]) for m in METHODS}

def tau_b(a, b):
    c = d = ta = tb = 0

    for i, j in combinations(range(len(a)), 2):
        da, db = np.sign(a[i] - a[j]), np.sign(b[i] - b[j])
        if da == 0 and db == 0:
            continue
        if da == 0:
            ta += 1
        elif db == 0:
            tb += 1
        elif da == db:
            c += 1
        else:
            d += 1
    den = math.sqrt((c + d + ta) * (c + d + tb))
    return float("nan") if den == 0 else (c - d) / den

def strict_reversals(pred, actual):
    wrong = comparable = 0

    for a, b in combinations(ELIGIBLE, 2):
        p, t = pred[a] - pred[b], actual[a] - actual[b]
        if p != 0 and t != 0:
            comparable += 1
            wrong += int(p * t < 0)
    return wrong, comparable

def in_best(fmt, ranks):
    return fmt in [f for f, r in ranks.items() if r == 1]


### 7.1 Chosen format and extra error

Show the initial choice, the final choice after any tie check, and whether the choice matches each reference's best formats. “Regret” means the extra full-output error compared with the reference's best choice.


In [9]:
rows = {}

for m in METHODS:
    r = RESULTS[m]
    row = {
        "Initial choice": r["raw_choice"],
        "Initial top group": " = ".join(r["top_ties"]),
        "Final choice": r["chosen"],
        "Tie check used": "✅" if r["refinement_scores"] else "-",
        "Optimal vs full baseline": "✅" if in_best(r["chosen"], base_rank) else "❌",
        "Regret vs full baseline (pp)": base_err[r["chosen"]] - min(base_err.values()),
    }
    if OPT:
        row["Optimal vs scale-optimised"] = "✅" if in_best(r["chosen"], opt_rank) else "❌"
        row["Regret vs scale-optimised (pp)"] = opt_err[r["chosen"]] - min(opt_err.values())
    rows[m] = row
rows["Full baseline (reference)"] = {"Final choice": " = ".join(BASE["ranking"][0])}
if OPT:
    rows["Scale-optimised (reference)"] = {"Final choice": " = ".join(OPT["ranking"][0])}
winner_matrix = pd.DataFrame(rows).T
winner_matrix.index.name = "Method"
fmt_cols = {c: "{:.4f}" for c in winner_matrix.columns if c.startswith("Regret")}
show_matrix(winner_matrix, "wt_x winner and selection regret", fmt=fmt_cols,
            flag_cols=[c for c in winner_matrix.columns if c.startswith("Optimal")])


,Initial choice,Initial top group,Final choice,Tie check used,Optimal vs full baseline,Regret vs full baseline (pp),Optimal vs scale-optimised,Regret vs scale-optimised (pp)
Method,,,,,,,,
wt_x,E2M1,E2M1,E2M1,-,✅,0.0000,✅,0.0000
Full baseline (reference),-,-,E2M1,-,-,-,-,-
Scale-optimised (reference),-,-,E2M1,-,-,-,-,-


### 7.2 Rankings

Compare each format's rank under selected-block scoring and the full-output references. Rank 1 is best.


In [10]:
rank_matrix = pd.DataFrame({**{m: [method_rank[m][f] for f in ELIGIBLE] for m in METHODS},
                            "Full baseline": [base_rank[f] for f in ELIGIBLE]}, index=idx_fmt)
if OPT:
    rank_matrix["Scale-optimised"] = [opt_rank[f] for f in ELIGIBLE]
rank_matrix.columns.name = "Method"
show_matrix(rank_matrix, "Format rank (1 = best, ties share a rank)", fmt="{:d}", best="min")


Method,wt_x,Full baseline,Scale-optimised
Format,,,
E2M1,1,1,1
E1M2,4,4,5
MXUINT4,5,5,2
MXINT4,2,2,2
BFP4,2,2,2


### 7.3 Scores and full errors

Show the selected-block scores alongside complete-matrix errors. These use different measurements, so their numerical values should not be compared directly.


In [11]:
score_matrix = pd.DataFrame({m: [RESULTS[m]["scores"][f] for f in ELIGIBLE] for m in METHODS}, index=idx_fmt)
score_matrix.columns.name = "Method"
show_matrix(score_matrix, "wt_x selected-block output-error score (use for ranking)",
            fmt="{:.6f}", best="min")

err_cols = {"Full baseline (%)": [base_err[f] for f in ELIGIBLE]}
if OPT:
    err_cols["Scale-optimised (%)"] = [opt_err[f] for f in ELIGIBLE]
    err_cols["Exponent-search gain (pp)"] = [base_err[f] - opt_err[f] for f in ELIGIBLE]
error_matrix = pd.DataFrame(err_cols, index=idx_fmt)
show_matrix(error_matrix, "Actual output error % of each format", fmt="{:.4f}",
            best="min")


Method,wt_x
Format,
E2M1,0.093291
E1M2,0.121475
MXUINT4,0.135618
MXINT4,0.121430
BFP4,0.121430


,Full baseline (%),Scale-optimised (%),Exponent-search gain (pp)
Format,,,
E2M1,12.5523,12.0307,0.5217
E1M2,15.8172,14.0042,1.8130
MXUINT4,17.1262,13.4521,3.6742
MXINT4,15.8080,13.4521,2.3559
BFP4,15.8080,13.4521,2.3559


### 7.4 Ranking agreement

Kendall tau-b measures agreement between complete rankings: 1 means agreement when the statistic is defined, and −1 means reversed order. Strict reversals count pairs of formats placed in opposite orders. A shared winner does not necessarily mean the complete rankings agree.


In [12]:
agree_rows = {}

for m in METHODS:
    pr = np.array([method_rank[m][f] for f in ELIGIBLE])
    br = np.array([base_rank[f] for f in ELIGIBLE])
    wrong_b, pairs_b = strict_reversals(method_rank[m], base_rank)
    row = {"tau vs full baseline": tau_b(pr, br),
           "Strict reversals vs full baseline": f"{wrong_b} / {pairs_b}",
           "Exact rank match vs full baseline": "✅" if (pr == br).all() else "❌"}
    if OPT:
        orr = np.array([opt_rank[f] for f in ELIGIBLE])
        wrong_o, pairs_o = strict_reversals(method_rank[m], opt_rank)
        row.update({"tau vs scale-optimised": tau_b(pr, orr),
                    "Strict reversals vs scale-optimised": f"{wrong_o} / {pairs_o}",
                    "Exact rank match vs scale-optimised": "✅" if (pr == orr).all() else "❌"})
    agree_rows[m] = row
agreement_matrix = pd.DataFrame(agree_rows).T
agreement_matrix.index.name = "Method"
show_matrix(agreement_matrix, "Ranking agreement with the references",
            fmt={c: "{:.3f}" for c in agreement_matrix.columns if c.startswith("tau")},
            flag_cols=[c for c in agreement_matrix.columns if c.startswith("Exact")])


,tau vs full baseline,Strict reversals vs full baseline,Exact rank match vs full baseline,tau vs scale-optimised,Strict reversals vs scale-optimised,Exact rank match vs scale-optimised
Method,,,,,,
wt_x,1.000,0 / 9,✅,0.630,1 / 7,❌


### 7.5 Selected work and arithmetic counts

Show how many blocks and weights were selected and how much importance they cover. Separate counts report initial selection, tie checks, and final full quantization. These are estimates under the notebook's counting model, not measured speedups.


In [13]:
brute_flops = flops(BASE["counts"])
cost_rows = {}

for m in METHODS:
    r = RESULTS[m]
    cost_rows[m] = {
        "Blocks used": f"{len(r['selected_blocks'])} / {r['n_blocks']}",
        "Weights used (%)": 100.0 * r["selected_weights"] / W64.size,
        "Retained importance (%)": 100.0 * r["energy_fraction"],
        "Initial selection FLOPs": flops(r["raw_counts"]),
        "Tie-check FLOPs": flops(r["refinement_counts"]),
        "Finish-chosen FLOPs": flops(r["materialization_counts"]),
        "Production FLOPs": flops(r["production_counts"]),
        "Full baseline FLOPs": brute_flops,
        "Production / full baseline (%)": 100.0 * flops(r["production_counts"]) / brute_flops,
        "Values quantized": r["production_counts"]["quantized_values"],
    }
cost_matrix = pd.DataFrame(cost_rows).T
cost_matrix.index.name = "Method"
show_matrix(cost_matrix, "Coverage and arithmetic cost (algorithmic counts, not timings)",
            fmt={"Weights used (%)": "{:.2f}", "Retained importance (%)": "{:.2f}",
                 "Initial selection FLOPs": "{:,.0f}", "Tie-check FLOPs": "{:,.0f}",
                 "Finish-chosen FLOPs": "{:,.0f}", "Production FLOPs": "{:,.0f}",
                 "Full baseline FLOPs": "{:,.0f}", "Production / full baseline (%)": "{:.2f}",
                 "Values quantized": "{:,.0f}"})


,Blocks used,Weights used (%),Retained importance (%),Initial selection FLOPs,Tie-check FLOPs,Finish-chosen FLOPs,Production FLOPs,Full baseline FLOPs,Production / full baseline (%),Values quantized
Method,,,,,,,,,,
wt_x,122 / 400,37.64,50.27,"10,104,815",0,"68,596","10,173,411","26,078,012",39.01,"25,056"


### 7.6 Summary

Summarize the selected format and its comparison with the enabled references.


In [14]:
for m in METHODS:
    r = RESULTS[m]
    line = (f"{m:<13} -> {r['chosen']:<8} | optimal vs full baseline: {winner_matrix.loc[m, 'Optimal vs full baseline']}"
            f"  (regret {winner_matrix.loc[m, 'Regret vs full baseline (pp)']:.4f} pp)")
    if OPT:
        line += (f" | vs scale-optimised: {winner_matrix.loc[m, 'Optimal vs scale-optimised']}"
                 f"  (regret {winner_matrix.loc[m, 'Regret vs scale-optimised (pp)']:.4f} pp)")
    line += f" | production FLOPs {cost_matrix.loc[m, 'Production / full baseline (%)']:.1f}% of full baseline"
    print(line)
print("Full baseline best  :", " = ".join(BASE["ranking"][0]))
if OPT:
    print("Scale-optimised best:", " = ".join(OPT["ranking"][0]))


wt_x          -> E2M1     | optimal vs full baseline: ✅  (regret 0.0000 pp) | vs scale-optimised: ✅  (regret 0.0000 pp) | production FLOPs 39.0% of full baseline
Full baseline best  : E2M1
Scale-optimised best: E2M1


## Notes and references

The importance rule is a practical selection method, not a guarantee of the best full-output format. All evaluations use calibration inputs; they do not measure performance on unseen test inputs. Results depend on the configured formats and scale rules.

Background sources:

- Open Compute Project, *Microscaling Formats (MX) Specification v1.0*: MX formats and E2M1.
- Wang et al., *Optimizing Large Language Model Training Using FP4 Quantization* (2025): E1M2.
- Köster et al., *Flexpoint* (2017): shared-exponent background.
- IEEE 754: nearest rounding with ties-to-even.
- Kendall, *A New Measure of Rank Correlation* (1938): rank correlation background.

The block-importance selection rule is a project-specific method.
